# Predictive Maintenance 4.0 — Physics-Aware Industrial Digital Twin

**Mechanical Engineering × Vibration Diagnostics × Thermal Modeling × Machine Learning × 3D Visualization**

This advanced portfolio notebook includes:
1. Multi-sensor synthetic machine operating data and vibration waveforms
2. Bearing geometry calculator (FTF, BPFO, BPFI, BSF)
3. FFT, envelope spectrum, RMS, crest factor, kurtosis and order features
4. Physics-inspired lumped thermal model with load-dependent heat generation and cooling
5. Physics-aware health indicators and operating-condition trends
6. Isolation Forest anomaly detection, fault classification and RUL regression
7. Machine-wise holdout validation and evaluation plots
8. Explainability through permutation importance
9. Drift monitoring between baseline and later operating windows
10. Interactive 3D machine digital twin, condition map and degradation animation
11. Model/data export and a small FastAPI inference service

> **Important:** all sensor data and simulated thermal behavior are synthetic and simplified. This is a reproducible portfolio demonstration, not a validated industrial diagnostic system. Never use it for safety-critical maintenance decisions.


## 1. Environment setup

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib plotly joblib fastapi uvicorn


## 2. Imports and reproducibility

In [ ]:
import json, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy.fft import rfft, rfftfreq
from scipy.signal import butter, sosfilt, hilbert
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest, RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED=42
rng=np.random.default_rng(SEED)
FS=4096
N=4096
t=np.arange(N)/FS
ART=Path("artifacts")
ART.mkdir(exist_ok=True)


## 3. Bearing geometry and characteristic frequencies

In [ ]:
def bearing_frequencies(rpm, rolling_elements, ball_diameter_mm,
                        pitch_diameter_mm, contact_angle_deg=0.0):
    # Standard idealized bearing kinematic frequency equations.
    fr=float(rpm)/60.0
    n=int(rolling_elements)
    ratio=(ball_diameter_mm/pitch_diameter_mm)*np.cos(np.deg2rad(contact_angle_deg))
    return {
        "shaft_frequency_hz": fr,
        "FTF_hz": 0.5*fr*(1-ratio),
        "BPFO_hz": 0.5*n*fr*(1-ratio),
        "BPFI_hz": 0.5*n*fr*(1+ratio),
        "BSF_hz": (pitch_diameter_mm/(2*ball_diameter_mm))*fr*(1-ratio**2)
    }
bearing_example=bearing_frequencies(1800, 8, 10, 50, 0)
pd.Series(bearing_example, name="frequency_hz")


## 4. Physics-inspired thermal model

In [ ]:
def simulate_temperature(load, ambient_c=25.0, duration_s=900, dt_s=1.0,
                          heat_capacity=18000.0, cooling_coeff=22.0):
    # Lumped first-order energy balance: C*dT/dt = Q_generated - h*(T-Tambient).
    # Illustrative normalized load-to-heat relationship; not a machine-specific model.
    steps=int(duration_s/dt_s)
    temp=np.empty(steps, dtype=float)
    temp[0]=ambient_c+3.0
    q_base=95.0 + 310.0*float(load)**2
    for k in range(1,steps):
        q=q_base*(1.0+0.035*np.sin(2*np.pi*k/(steps/3)))
        dT=(q-cooling_coeff*(temp[k-1]-ambient_c))/heat_capacity*dt_s
        temp[k]=temp[k-1]+dT
    return temp
thermal_demo=simulate_temperature(load=1.0)
plt.figure(figsize=(9,3))
plt.plot(np.arange(len(thermal_demo)), thermal_demo)
plt.title("Synthetic lumped thermal response")
plt.xlabel("Time (s)"); plt.ylabel("Temperature (°C)")
plt.grid(True); plt.show()


## 5. Generate multi-machine synthetic sensor and waveform data

In [ ]:
MACHINES, CYCLES = 5, 220
fault_classes=["healthy","imbalance","misalignment","bearing_defect"]
records, waveforms, thermal_traces = [], [], []
for mi in range(MACHINES):
    machine_bias=rng.normal(0,0.8)
    for cyc in range(CYCLES):
        degradation=cyc/(CYCLES-1)
        rpm=float(rng.normal(1800+mi*9,22))
        load=float(rng.uniform(0.60,1.25))
        shaft_hz=rpm/60.0
        probs=[max(.20,.62-.40*degradation),.13,.12,min(.55,.13+.40*degradation)]
        probs=np.array(probs); probs=probs/probs.sum()
        fault=str(rng.choice(fault_classes,p=probs))
        ambient=25.0
        thermal=simulate_temperature(load, ambient_c=ambient, duration_s=180, dt_s=1.0)
        temp=float(thermal[-1]+22*degradation+machine_bias+rng.normal(0,0.8))
        amp=(.20+1.0*degradation)*load
        x=amp*np.sin(2*np.pi*shaft_hz*t)
        x+=.16*amp*np.sin(2*np.pi*2*shaft_hz*t+.3)
        x+=(.04+.15*degradation)*np.sin(2*np.pi*185*t)
        if fault=="imbalance":
            x+=.65*np.sin(2*np.pi*shaft_hz*t+.12)
        elif fault=="misalignment":
            x+=.62*np.sin(2*np.pi*2*shaft_hz*t)+.28*np.sin(2*np.pi*3*shaft_hz*t)
        elif fault=="bearing_defect":
            impact=np.zeros(N)
            interval=max(1,int(FS/(3.6*shaft_hz)))
            impact[np.arange(0,N,interval)]=1.0
            tau=np.arange(180)
            impulse=np.exp(-tau/28)*np.sin(2*np.pi*720*tau/FS)
            x+=.72*np.convolve(impact,impulse,mode="full")[:N]
        x+=rng.normal(0,.09+.06*degradation,N)
        records.append(dict(machine_id=f"UNIT_{mi+1}",cycle=cyc,rpm=rpm,load=load,
            ambient_c=ambient,temperature_c=temp,degradation=degradation,fault_type=fault))
        waveforms.append(x)
        thermal_traces.append(thermal)
raw=pd.DataFrame(records)
print("Rows:",len(raw)," | Machines:",raw.machine_id.nunique())
display(raw.head())


## 6. Signal feature extraction — time, FFT, envelope and rotational orders

In [ ]:
def extract_features(x, rpm, fs=FS):
    rms=float(np.sqrt(np.mean(x*x)))
    peak=float(np.max(np.abs(x)))
    freq=rfftfreq(len(x),1/fs)
    spec=np.abs(rfft(x))
    # Envelope analysis on a selected illustrative resonance band.
    sos=butter(4,[500,1400],btype="bandpass",fs=fs,output="sos")
    band=sosfilt(sos,x)
    env=np.abs(hilbert(band))
    env_freq=rfftfreq(len(env),1/fs)
    env_spec=np.abs(rfft(env-env.mean()))
    out={
        "vibration_rms":rms,"vibration_peak":peak,
        "peak_to_peak":float(np.ptp(x)),
        "crest_factor":float(peak/(rms+1e-9)),
        "kurtosis":float(kurtosis(x)),
        "skewness":float(skew(x)),
        "dominant_frequency_hz":float(freq[1:][np.argmax(spec[1:])]),
        "envelope_rms":float(np.sqrt(np.mean(env**2))),
        "envelope_peak_amplitude":float(env_spec[1:][np.argmax(env_spec[1:])])
    }
    for lo,hi in [(0,80),(80,200),(200,500),(500,1000),(1000,2000)]:
        mask=(freq>=lo)&(freq<hi)
        out[f"fft_energy_{lo}_{hi}"]=float(np.sum(spec[mask]**2))
    for order in (1,2,3):
        idx=int(np.argmin(np.abs(freq-order*(rpm/60))))
        out[f"order_{order}x_amplitude"]=float(spec[idx])
    return out, (freq,spec,env_freq,env_spec)
feature_rows=[]
spectra={}
for i,(sig,rpm) in enumerate(zip(waveforms,raw.rpm)):
    f,sp=extract_features(sig,rpm)
    feature_rows.append(f)
    if i in (0,220,440,660,880):
        spectra[i]=sp
df=pd.concat([raw.reset_index(drop=True),pd.DataFrame(feature_rows)],axis=1)
df["rul_cycles"]=(1-df.degradation)*(CYCLES-1)
display(df.head())


## 7. Vibration FFT and envelope spectrum

In [ ]:
sample_idx=list(spectra)[-1]
freq,spec,env_freq,env_spec=spectra[sample_idx]
fig=go.Figure()
fig.add_trace(go.Scatter(x=freq,y=spec,name="FFT"))
fig.update_layout(title=f"Vibration FFT — sample {sample_idx}",
                  xaxis_title="Frequency (Hz)",yaxis_title="Amplitude")
fig.show()
fig=go.Figure()
fig.add_trace(go.Scatter(x=env_freq,y=env_spec,name="Envelope spectrum"))
fig.update_layout(title="Envelope spectrum — synthetic bearing impact example",
                  xaxis_title="Frequency (Hz)",yaxis_title="Amplitude")
fig.show()


## 8. Order tracking and condition trends

In [ ]:
df["shaft_hz"]=df.rpm/60.0
df["dominant_order"]=df.dominant_frequency_hz/(df.shaft_hz+1e-9)
px.scatter(df,x="cycle",y="dominant_order",color="fault_type",
           facet_col="machine_id",title="Dominant spectral order by machine").show()
px.line(df,x="cycle",y="temperature_c",color="machine_id",
        title="Operating temperature trend").show()


## 9. Physics-aware Health Index

In [ ]:
def minmax(s):
    return (s-s.min())/(s.max()-s.min()+1e-9)
# Transparent illustrative score. Calibration is required for real equipment.
df["health_index"]=(100*(1-(.42*minmax(df.vibration_rms)+
    .20*minmax(df.temperature_c)+.18*minmax(df.kurtosis.clip(lower=0))+
    .12*minmax(df.envelope_rms)+.08*minmax(df.order_2x_amplitude)))).clip(0,100)
px.line(df,x="cycle",y="health_index",color="machine_id",
        title="Machine Health Index (illustrative 0–100)").show()


## 10. Interactive 3D machine condition map

In [ ]:
fig=px.scatter_3d(df,x="vibration_rms",y="temperature_c",z="envelope_rms",
    color="health_index",symbol="fault_type",
    hover_data=["machine_id","cycle","rpm","load","rul_cycles"],
    color_continuous_scale="Turbo",title="3D Machine Condition Space")
fig.update_layout(scene=dict(xaxis_title="Vibration RMS",
    yaxis_title="Temperature (°C)",zaxis_title="Envelope RMS"))
fig.show()


## 11. Conceptual 3D rotating machinery digital twin

In [ ]:
theta=np.linspace(0,2*np.pi,100)
fig=go.Figure()
# Shaft centerline
fig.add_trace(go.Scatter3d(x=np.linspace(-4,4,120),y=np.zeros(120),z=np.zeros(120),
    mode="lines",line=dict(width=16),name="Shaft"))
# Rotor rings
for xpos,label in [(-1.5,"Rotor A"),(1.5,"Rotor B")]:
    fig.add_trace(go.Scatter3d(x=np.full_like(theta,xpos),y=.9*np.cos(theta),
        z=.9*np.sin(theta),mode="lines",line=dict(width=10),name=label))
# Bearing support blocks as translucent boxes
for xpos,label in [(-3.1,"Bearing support 1"),(3.1,"Bearing support 2")]:
    fig.add_trace(go.Mesh3d(
        x=[xpos-.35,xpos+.35,xpos+.35,xpos-.35,xpos-.35,xpos+.35,xpos+.35,xpos-.35],
        y=[-.7,-.7,.7,.7,-.7,-.7,.7,.7],
        z=[-.55,-.55,-.55,-.55,.55,.55,.55,.55],
        i=[0,0,4,4,0,0,1,1,2,2,3,3],
        j=[1,2,5,6,1,5,2,6,3,7,0,4],
        k=[2,3,6,7,5,4,6,5,7,6,4,7],
        opacity=.38,name=label))
fig.update_layout(title="Conceptual 3D Rotating Machinery Digital Twin",
    scene=dict(xaxis_title="Shaft axis",yaxis_title="Y",zaxis_title="Z",aspectmode="data"))
fig.show()


## 12. Animated 3D degradation trajectory

In [ ]:
unit=df[df.machine_id=="UNIT_1"].sort_values("cycle")
anim=px.scatter_3d(unit,x="vibration_rms",y="temperature_c",z="envelope_rms",
    color="health_index",animation_frame="cycle",
    range_x=[df.vibration_rms.min(),df.vibration_rms.max()],
    range_y=[df.temperature_c.min(),df.temperature_c.max()],
    range_z=[df.envelope_rms.min(),df.envelope_rms.max()],
    title="Animated 3D Degradation Trajectory — UNIT_1")
anim.show()


## 13. Anomaly detection — Isolation Forest

In [ ]:
feature_cols=[c for c in df.columns if c.startswith(("vibration_","fft_energy_","order_","envelope_"))]
feature_cols += ["rpm","load","temperature_c","peak_to_peak","crest_factor",
                 "kurtosis","skewness","dominant_frequency_hz","dominant_order"]
X=df[feature_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
healthy=X[df.fault_type=="healthy"]
anomaly_model=make_pipeline(StandardScaler(),
    IsolationForest(n_estimators=350,contamination=.08,random_state=SEED))
anomaly_model.fit(healthy)
df["anomaly_label"]=anomaly_model.predict(X)
df["anomaly_score"]=-anomaly_model.decision_function(X)
print(df.anomaly_label.value_counts().rename({1:"inlier",-1:"potential anomaly"}))
px.scatter_3d(df,x="vibration_rms",y="temperature_c",z="envelope_rms",
    color=df.anomaly_label.astype(str),hover_data=["machine_id","cycle","fault_type"],
    title="3D Anomaly Detection View").show()


## 14. Fault classification and RUL regression — machine holdout

In [ ]:
train_mask=df.machine_id!="UNIT_5"
test_mask=~train_mask
X_train,X_test=X[train_mask],X[test_mask]
fault_model=make_pipeline(StandardScaler(),
    RandomForestClassifier(n_estimators=350,class_weight="balanced",
                           min_samples_leaf=2,random_state=SEED))
fault_model.fit(X_train,df.loc[train_mask,"fault_type"])
fault_pred=fault_model.predict(X_test)
print(classification_report(df.loc[test_mask,"fault_type"],fault_pred,zero_division=0))
labels=sorted(df.fault_type.unique())
cm=confusion_matrix(df.loc[test_mask,"fault_type"],fault_pred,labels=labels)
px.imshow(cm,x=labels,y=labels,text_auto=True,
          title="Fault Classification — Holdout Machine UNIT_5").show()
rul_model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,
                                random_state=SEED,n_jobs=-1)
rul_model.fit(X_train,df.loc[train_mask,"rul_cycles"])
rul_pred=np.maximum(0,rul_model.predict(X_test))
print("RUL MAE:",round(mean_absolute_error(df.loc[test_mask,"rul_cycles"],rul_pred),2))
print("RUL RMSE:",round(mean_squared_error(df.loc[test_mask,"rul_cycles"],rul_pred)**.5,2))
print("RUL R2:",round(r2_score(df.loc[test_mask,"rul_cycles"],rul_pred),3))


## 15. Model explainability — permutation importance

In [ ]:
perm=permutation_importance(rul_model,X_test,df.loc[test_mask,"rul_cycles"],
    n_repeats=5,random_state=SEED,scoring="neg_mean_absolute_error")
importance=pd.Series(perm.importances_mean,index=feature_cols).sort_values().tail(15)
importance.plot(kind="barh",figsize=(9,5),title="RUL Model — Permutation Importance")
plt.tight_layout(); plt.show()


## 16. Data drift monitoring

In [ ]:
# Compare early-life baseline to late-life window using standardized mean shift.
baseline=df[df.cycle < CYCLES*.25][feature_cols]
current=df[df.cycle > CYCLES*.75][feature_cols]
drift=pd.DataFrame({
    "baseline_mean":baseline.mean(),
    "current_mean":current.mean(),
    "standardized_shift":(current.mean()-baseline.mean())/(baseline.std()+1e-9)
})
drift["abs_shift"]=drift.standardized_shift.abs()
display(drift.sort_values("abs_shift",ascending=False).head(15))
px.bar(drift.reset_index(),x="index",y="standardized_shift",
       title="Feature Distribution Shift: Early vs Late Operating Life").show()


## 17. Export models, dataset and metadata

In [ ]:
df.to_csv(ART/"synthetic_sensor_dataset.csv",index=False)
joblib.dump(anomaly_model,ART/"anomaly_model.joblib")
joblib.dump(fault_model,ART/"fault_classifier.joblib")
joblib.dump(rul_model,ART/"rul_regressor.joblib")
(ART/"feature_columns.json").write_text(json.dumps(feature_cols,indent=2))
(ART/"bearing_frequencies_example.json").write_text(json.dumps(bearing_example,indent=2))
(ART/"metadata.json").write_text(json.dumps({
    "project":"Predictive Maintenance 4.0",
    "synthetic_data":True,"sampling_frequency_hz":FS,
    "samples_per_snapshot":N,"train_machines":["UNIT_1","UNIT_2","UNIT_3","UNIT_4"],
    "holdout_machine":"UNIT_5","fault_classes":fault_classes,
    "thermal_model":"illustrative lumped first-order energy balance"
},indent=2))
print("Exported dataset, fitted models and metadata to:",ART.resolve())


## 18. Engineering interpretation and limitations
- Bearing characteristic frequencies depend on actual bearing geometry, contact angle, slip and load; ideal formulas are only a first estimate.
- Envelope analysis needs a machine-appropriate resonance band, sensor mounting location and sampling frequency.
- True order tracking under variable speed requires a tachometer and angular resampling; this notebook uses spectral frequency normalized by nominal shaft frequency.
- The thermal model is a lumped illustrative energy balance, not CFD or a validated thermal simulation.
- Health Index weights are illustrative and require calibration against machine-specific failure history.
- RUL labels are synthetic and derived from an artificial degradation trajectory. Real RUL requires run-to-failure data and a defensible failure definition.
- Machine-wise holdout reduces machine identity leakage, but synthetic data cannot establish real-world generalization.
- Do not use this notebook for operational maintenance or safety decisions without sensor calibration, machine-specific validation and engineering approval.

## 3D Engineering Visualization Suite — v5

This section adds interactive, rotatable Plotly 3D engineering diagrams:
1. CAD-style shaft, rotor, bearing and housing assembly
2. Exploded bearing component view
3. Shaft stress/deformation *illustrative* contour
4. 3D vibration waveform / spectrum view
5. Machine digital-twin health map with fault severity
6. Animated degradation from healthy to critical

**Important:** These are conceptual/synthetic visualizations for portfolio demonstration. They are not manufacturing drawings, validated FEA, or measured machine data. Replace dimensions and sensor values with verified measurements before engineering decisions.


In [ ]:
# Interactive 3D diagram helpers
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _cylinder_mesh(radius, x0, x1, n=48, m=20):
    theta = np.linspace(0, 2*np.pi, n)
    x = np.linspace(x0, x1, m)
    T, X = np.meshgrid(theta, x)
    Y, Z = radius*np.cos(T), radius*np.sin(T)
    return X, Y, Z

def add_cylinder(fig, radius, x0, x1, color, name, opacity=1.0):
    X, Y, Z = _cylinder_mesh(radius, x0, x1)
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, surfacecolor=np.zeros_like(X),
        colorscale=[[0,color],[1,color]], showscale=False, opacity=opacity,
        name=name, hovertemplate=f"{name}<extra></extra>"))

def add_ring(fig, x, r_inner, r_outer, color, name, opacity=1.0):
    theta = np.linspace(0,2*np.pi,64)
    xx = np.array([x-0.12,x+0.12])
    T,X = np.meshgrid(theta,xx)
    Y1,Z1 = r_outer*np.cos(T),r_outer*np.sin(T)
    Y0,Z0 = r_inner*np.cos(T),r_inner*np.sin(T)
    fig.add_trace(go.Surface(x=X,y=Y1,z=Z1,colorscale=[[0,color],[1,color]],
        showscale=False,opacity=opacity,name=name,hovertemplate=f"{name}<extra></extra>"))
    fig.add_trace(go.Surface(x=X,y=Y0,z=Z0,colorscale=[[0,color],[1,color]],
        showscale=False,opacity=opacity,showslegend=False,hoverinfo="skip"))

def engineering_assembly_3d():
    f=go.Figure()
    add_cylinder(f,0.16,-3.0,3.0,"#9aa7b5","Drive shaft")
    add_cylinder(f,0.78,-0.48,0.48,"#d97706","Rotor disk")
    for x in [-2.0,2.0]:
        add_ring(f,x,0.22,0.52,"#64748b","Bearing housing")
        # inner race
        add_ring(f,x,0.16,0.25,"#38bdf8","Bearing inner race")
        # bearing balls
        a=np.linspace(0,2*np.pi,10,endpoint=False)
        f.add_trace(go.Scatter3d(x=np.full(10,x),y=.36*np.cos(a),z=.36*np.sin(a),
            mode="markers",marker=dict(size=7,color="#f8fafc",line=dict(color="#334155",width=1)),
            name="Rolling elements"))
    f.add_trace(go.Scatter3d(x=[-3.1,3.1],y=[0,0],z=[0,0],mode="markers+text",
        text=["Coupling end","Load end"],textposition="top center",
        marker=dict(size=4,color="#ef4444"),name="Shaft ends"))
    f.update_layout(title="Interactive 3D Machine Assembly — conceptual CAD view",
        scene=dict(xaxis_title="Axial position",yaxis_title="Y",zaxis_title="Z",
                   aspectmode="data"),height=650,legend=dict(orientation="h"))
    return f

engineering_assembly_3d().show()


In [ ]:
def bearing_exploded_3d():
    f=go.Figure()
    # Axial exploded view: outer ring, inner ring and rolling elements are separated.
    for x,ri,ro,col,label in [
        (-0.8,0.58,0.86,"#64748b","Outer race"),
        (0.0,0.25,0.48,"#0ea5e9","Inner race")]:
        add_ring(f,x,ri,ro,col,label)
    ang=np.linspace(0,2*np.pi,12,endpoint=False)
    f.add_trace(go.Scatter3d(x=np.full(len(ang),0.42),y=.57*np.cos(ang),z=.57*np.sin(ang),
        mode="markers",marker=dict(size=10,color="#f59e0b",line=dict(color="#78350f",width=1)),
        name="Rolling elements"))
    f.add_trace(go.Scatter3d(x=[1.0],y=[0],z=[0],mode="markers+text",
        text=["Cage (schematic)"],textposition="top center",
        marker=dict(size=9,color="#22c55e"),name="Cage reference"))
    f.update_layout(title="Exploded 3D Bearing Components",
        scene=dict(xaxis_title="Exploded axial offset",yaxis_title="Y",zaxis_title="Z",
                   aspectmode="data"),height=600)
    return f

bearing_exploded_3d().show()


In [ ]:
def shaft_stress_concept_3d():
    # Illustrative stress field only: smooth axial/circumferential pattern, not FEA.
    theta=np.linspace(0,2*np.pi,100)
    x=np.linspace(-3,3,100)
    T,X=np.meshgrid(theta,x)
    radius=0.35
    Y=radius*np.cos(T); Z=radius*np.sin(T)
    stress=35 + 55*np.exp(-((X-0.7)/0.65)**2)*(0.65+0.35*np.cos(2*T))
    f=go.Figure(go.Surface(x=X,y=Y,z=Z,surfacecolor=stress,
        colorscale="Turbo",colorbar=dict(title="Illustrative<br>stress (MPa)"),
        customdata=stress,hovertemplate="x=%{x:.2f}<br>stress=%{customdata:.1f} MPa<extra></extra>"))
    f.update_layout(title="Shaft Stress Contour — illustrative visualization, not validated FEA",
        scene=dict(xaxis_title="Shaft axis",yaxis_title="Y",zaxis_title="Z",aspectmode="data"),
        height=620)
    return f

shaft_stress_concept_3d().show()


In [ ]:
def vibration_3d_view(rpm=1800, duration=0.12, sample_rate=4000):
    t=np.arange(0,duration,1/sample_rate)
    f1=rpm/60
    signal=0.7*np.sin(2*np.pi*f1*t)+0.25*np.sin(2*np.pi*4.2*f1*t)
    signal += 0.12*np.sin(2*np.pi*9*f1*t)*np.sin(2*np.pi*13*t)
    # A 3D ribbon-like trace: time, instantaneous amplitude, running phase.
    phase=np.mod(2*np.pi*f1*t,2*np.pi)
    fig=go.Figure(go.Scatter3d(x=t,y=signal,z=phase,mode="lines",
        line=dict(color=np.abs(signal),colorscale="Viridis",width=5),
        name="Synthetic vibration"))
    fig.update_layout(title=f"3D Vibration Trace — synthetic signal at {rpm} RPM",
        scene=dict(xaxis_title="Time (s)",yaxis_title="Acceleration (a.u.)",
                   zaxis_title="Shaft phase (rad)"),height=600)
    return fig

vibration_3d_view().show()


In [ ]:
def digital_twin_health_3d():
    # Synthetic operating points: RPM, temperature and vibration RMS.
    rng=np.random.default_rng(42)
    n=180
    rpm=rng.uniform(900,3000,n)
    temp=35+0.012*(rpm-900)+rng.normal(0,3,n)
    vib=0.8+0.00055*(rpm-900)+rng.normal(0,0.18,n)
    severity=np.clip((temp-40)/30 + (vib-0.8)/2.5,0,1)
    labels=np.where(severity<0.35,"Healthy",np.where(severity<0.68,"Warning","Critical"))
    fig=go.Figure(go.Scatter3d(x=rpm,y=temp,z=vib,mode="markers",
        marker=dict(size=5,color=severity,colorscale="RdYlGn_r",
                    cmin=0,cmax=1,colorbar=dict(title="Fault<br>severity"),
                    line=dict(width=0.3,color="#334155")),
        text=labels,customdata=np.column_stack([severity,labels]),
        hovertemplate="RPM=%{x:.0f}<br>Temp=%{y:.1f} °C<br>Vibration RMS=%{z:.2f}<br>Severity=%{customdata[0]:.2f}<br>%{customdata[1]}<extra></extra>"))
    fig.update_layout(title="3D Digital Twin — operating-condition health map (synthetic)",
        scene=dict(xaxis_title="RPM",yaxis_title="Temperature (°C)",
                   zaxis_title="Vibration RMS (a.u.)"),height=650)
    return fig

digital_twin_health_3d().show()


In [ ]:
def degradation_animation_3d():
    cycles=np.arange(0,101,5)
    health=np.clip(100-0.82*cycles-0.0015*cycles**2,5,100)
    temp=35+0.24*cycles
    vib=0.5+0.025*cycles+0.0007*cycles**2
    frames=[]
    for i,c in enumerate(cycles):
        frames.append(go.Frame(name=str(c),data=[go.Scatter3d(
            x=cycles[:i+1],y=health[:i+1],z=vib[:i+1],mode="lines+markers",
            line=dict(width=6,color="#f97316"),marker=dict(size=4,color="#0ea5e9"))],
            traces=[0]))
    fig=go.Figure(data=[go.Scatter3d(x=[cycles[0]],y=[health[0]],z=[vib[0]],
        mode="lines+markers",line=dict(width=6,color="#f97316"))],frames=frames)
    fig.update_layout(title="Animated Machine Degradation — synthetic trend",
        scene=dict(xaxis_title="Operating cycles",yaxis_title="Health index (%)",
                   zaxis_title="Vibration RMS (a.u.)"),
        updatemenus=[dict(type="buttons",showactive=False,
            buttons=[dict(label="Play",method="animate",args=[None,{"frame":{"duration":350,"redraw":True},
                "fromcurrent":True}]),dict(label="Pause",method="animate",
                args=[[None],{"mode":"immediate","frame":{"duration":0,"redraw":False}}])])],
        height=650)
    return fig

degradation_animation_3d().show()


### Export interactive diagrams as standalone HTML

Run this cell to save a shareable HTML gallery. The files can be opened in a browser and committed to GitHub. Plotly diagrams remain interactive (rotate/zoom/pan).


In [ ]:
from pathlib import Path
gallery_dir=Path("outputs/gallery")
gallery_dir.mkdir(parents=True,exist_ok=True)
diagram_builders={
    "01_machine_assembly_3d": engineering_assembly_3d,
    "02_bearing_exploded_3d": bearing_exploded_3d,
    "03_shaft_stress_concept_3d": shaft_stress_concept_3d,
    "04_vibration_3d": vibration_3d_view,
    "05_digital_twin_health_3d": digital_twin_health_3d,
    "06_degradation_animation_3d": degradation_animation_3d,
}
for filename,builder in diagram_builders.items():
    builder().write_html(gallery_dir/f"{filename}.html",include_plotlyjs="cdn")
print(f"Saved {len(diagram_builders)} interactive 3D diagrams to {gallery_dir.resolve()}")


## v6 Advanced Engineering Layer

New features: parametric 3D shaft geometry, 3D operating envelope, transparent maintenance-priority scoring, sensor-quality audit, and standalone HTML exports. All sample values are synthetic; limits must be replaced with OEM/site-approved values before operational use.


In [ ]:
def parametric_shaft_3d(length_mm=600, diameter_mm=40, shoulder_diameter_mm=65, shoulder_width_mm=70):
    import numpy as np, plotly.graph_objects as go
    L,d,ds,w=map(float,(length_mm,diameter_mm,shoulder_diameter_mm,shoulder_width_mm))
    if not (L>2*w and d>0 and ds>=d and w>0):
        raise ValueError("Require length > 2*shoulder_width and shoulder_diameter >= diameter > 0.")
    theta=np.linspace(0,2*np.pi,72); x=np.linspace(-L/2,L/2,140)
    X,T=np.meshgrid(x,theta)
    R=np.where(np.abs(X)>L/2-w,ds/2,d/2)
    Y,Z=R*np.cos(T),R*np.sin(T)
    fig=go.Figure(go.Surface(x=X,y=Y,z=Z,surfacecolor=R,colorscale="Blues",showscale=False,
        hovertemplate="Axial=%{x:.1f} mm<extra>Shaft</extra>"))
    fig.update_layout(title="Parametric 3D Shaft — conceptual dimensioned model",
        scene=dict(xaxis_title="Axial length (mm)",yaxis_title="Y (mm)",zaxis_title="Z (mm)",aspectmode="data"),height=620)
    return fig
parametric_shaft_3d().show()


In [ ]:
def operating_envelope_3d(seed=17, points=260):
    import numpy as np, plotly.graph_objects as go
    rng=np.random.default_rng(seed); rpm=rng.uniform(600,3600,points); load=rng.uniform(10,100,points)
    vib=.35+.00018*rpm+.006*load+rng.normal(0,.16,points)
    zone=np.where(vib<1.15,"Normal",np.where(vib<1.65,"Caution","Investigate"))
    colors={"Normal":"#16a34a","Caution":"#eab308","Investigate":"#dc2626"}
    fig=go.Figure()
    for label in colors:
        m=zone==label
        fig.add_trace(go.Scatter3d(x=rpm[m],y=load[m],z=vib[m],mode="markers",name=label,
            marker=dict(size=4,color=colors[label],opacity=.8)))
    fig.update_layout(title="3D Operating Envelope — illustrative vibration zones",
        scene=dict(xaxis_title="Speed (RPM)",yaxis_title="Load (%)",zaxis_title="Vibration RMS (a.u.)"),height=640)
    return fig
operating_envelope_3d().show()


In [ ]:
def maintenance_priority_score(vibration_rms, temperature_c, anomaly_flag=0,
                               vibration_limit=2.0, temperature_limit=90.0):
    import numpy as np
    v=np.clip(float(vibration_rms)/float(vibration_limit),0,1.5)/1.5
    t=np.clip((float(temperature_c)-25)/(float(temperature_limit)-25),0,1)
    a=1.0 if int(anomaly_flag) else 0.0
    score=100*(.50*v+.30*t+.20*a)
    band="Routine" if score<30 else ("Plan inspection" if score<60 else "Priority inspection")
    return {"priority_score":round(float(score),1),"priority_band":band,
            "factor_vibration":round(float(v),3),"factor_temperature":round(float(t),3),"factor_anomaly":a}
print(maintenance_priority_score(1.8,78,1))


In [ ]:
def sensor_quality_audit(dataframe, machine_col="machine_id", time_col="timestamp"):
    import pandas as pd
    df=dataframe.copy()
    report={"rows":int(len(df)),"columns":int(len(df.columns)),
            "duplicate_rows":int(df.duplicated().sum()),
            "missing_by_column":df.isna().sum().astype(int).to_dict(),
            "machine_count":int(df[machine_col].nunique()) if machine_col in df else None}
    if time_col in df:
        report["valid_timestamps"]=int(pd.to_datetime(df[time_col],errors="coerce").notna().sum())
    return report
print("Use sensor_quality_audit(your_sensor_dataframe) to audit a sensor table.")


### Reproducibility and validation

Split by machine/unit before model fitting to reduce leakage. Fit preprocessing only on training data. Report held-out machine precision, recall, F1, confusion matrix, MAE and RMSE. Synthetic-data scores are not evidence of real-world predictive performance. Thresholds and score weights require engineering validation.


In [ ]:
from pathlib import Path
v6_gallery=Path("outputs/gallery_v6"); v6_gallery.mkdir(parents=True,exist_ok=True)
parametric_shaft_3d().write_html(v6_gallery/"07_parametric_shaft_3d.html",include_plotlyjs="cdn")
operating_envelope_3d().write_html(v6_gallery/"08_operating_envelope_3d.html",include_plotlyjs="cdn")
print("Saved v6 interactive HTML diagrams:",v6_gallery.resolve())


## v7 Industrial Monitoring & Maintenance Workflow

This release adds a reproducible synthetic sensor-stream simulator, multi-machine fleet visualization, alert triage, and draft maintenance work-order generation. These are software demonstrations only; they do not connect to PLC/SCADA/IoT hardware or issue real work orders.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "monitoring.py").exists() and (PROJECT_ROOT.parent / "monitoring.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from monitoring import generate_sensor_batch, score_maintenance, build_work_orders

stream_batch = generate_sensor_batch(batch_id=3, machines=8)
stream_batch["priority_score"], stream_batch["priority"] = zip(
    *stream_batch.apply(score_maintenance, axis=1)
)
display(stream_batch)


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fleet_fig = px.scatter_3d(
    stream_batch, x="rpm", y="temperature_c", z="vibration_rms",
    color="priority", symbol="machine_id", hover_name="machine_id",
    title="v7 Multi-Machine Fleet Digital Twin — synthetic telemetry",
    labels={"rpm":"Speed (RPM)", "temperature_c":"Temperature (°C)",
            "vibration_rms":"Vibration RMS (a.u.)"}
)
fleet_fig.update_layout(height=650)
fleet_fig.show()


In [ ]:
work_orders = build_work_orders(stream_batch)
print(f"Draft work orders: {len(work_orders)}")
display(work_orders)


In [ ]:
# Export the synthetic batch and draft work queue for portfolio review
from pathlib import Path
export_dir = Path("outputs")
export_dir.mkdir(parents=True, exist_ok=True)
stream_batch.to_csv(export_dir / "v7_synthetic_sensor_batch.csv", index=False)
work_orders.to_csv(export_dir / "v7_draft_maintenance_queue.csv", index=False)
fleet_fig.write_html(export_dir / "v7_fleet_digital_twin.html", include_plotlyjs="cdn")
print("Exported v7 sensor batch, draft work queue, and interactive fleet map.")
